# SS 2pt TGEVP Template

This notebook is a runnable example of the existing TGEVP workflow using real example CSV data shipped in the repository.
Edit the user-input cell, validate the resulting plain-text style config, and then call the same backend used by the CLI.


## Imports / Setup

Run this notebook from the repository root, or adjust `REPO_ROOT` below.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from lqcd_analysis.notebook_workflows import (
    pretty_print_config,
    render_tgevp_input_text,
    run_tgevp_from_notebook,
    validate_tgevp_notebook_config,
)


## User Inputs

These fields intentionally mirror the current plain-text TGEVP input file.
The default paths point to tracked example data inside `examples/data/`.


In [ ]:
EXAMPLE_DATA = REPO_ROOT / "examples" / "data" / "l64c64a076_m140" / "comb_c2pt_csv"
EXAMPLE_OUTPUTS = REPO_ROOT / "examples" / "outputs" / "tgevp_notebook"

workflow_config = {
    "title_pattern": "l64c64a076_m140_SS_k0_pz*",
    "ns": 64,
    "nt": 64,
    "lattice_spacing_fm": 0.076,
    "c2pt": str(EXAMPLE_DATA / "c2pt_5_5_k0_pz*_real.csv"),
    "pzlist": [0],
    "fold_t": "periodic",
    "tsrange": [0, 20],
    "binsize": 1,
    "bootstrap_samples": 64,
    "bootstrap_size": 64,
    "seed": 2026,
    "results_dir": str(EXAMPLE_OUTPUTS),
}


## Option Guide

Edit only `workflow_config` in the cell above for normal usage.

- `title_pattern`: Output title pattern. Use `*` where the momentum index `pz` should be inserted, for example `demo_pz*`.
- `ns`: Spatial lattice extent `Ns`. Must match the dataset metadata.
- `nt`: Temporal lattice extent `Nt`. Must match the number of time slices in the correlator file.
- `lattice_spacing_fm`: Lattice spacing in fm. This is stored in metadata and summaries.
- `c2pt`: Correlator CSV path or wildcard pattern. Use a repository-relative path when possible. For multiple momenta, keep the `*` placeholder in the filename.
- `pzlist`: List of momentum indices to run, for example `[0]` or `[0, 1, 2]`.
- `fold_t`: Time-folding choice used to improve signal quality with boundary conditions.
  Choices: `"none"` or `False` = no folding; `"periodic"` or `True` = average `t` and `Nt-t`; `"antiperiodic"` = antisymmetric fold.
- `tsrange`: Two integers `[t_start, t_end]` selecting the time window used to build the TGEVP analysis.
- `binsize`: Integer bin size for configuration binning before bootstrap. Use `1` for no binning.
- `bootstrap_samples`: Number of bootstrap resamples. Larger values are slower but give smoother uncertainty estimates.
- `bootstrap_size`: Number of binned configurations drawn per bootstrap sample. `None` lets the backend use its default choice.
- `seed`: Random seed for bootstrap reproducibility.
- `results_dir`: Output directory. If omitted or set to `None`, outputs go to the notebook working directory.

Practical note:
- Increase `tsrange[1]` if you want to explore larger TGEVP orders. The current implementation needs enough time slices to build the Krylov matrices.


## Input Summary / Validation

This shows one unified notebook config object, plus the equivalent plain-text input content used by the legacy workflow.
Runner-only options such as `results_dir` stay in the same config block, but are omitted automatically from the rendered input text.


In [ ]:
print(pretty_print_config(workflow_config))
print(render_tgevp_input_text(workflow_config))
parsed_tgevp = validate_tgevp_notebook_config(workflow_config)
parsed_tgevp


## Run Analysis


In [ ]:
tgevp_outputs = run_tgevp_from_notebook(workflow_config)
for path in tgevp_outputs:
    print(path)


## Inspect Outputs

Outputs are written under `examples/outputs/`, which is intentionally ignored by git.


In [ ]:
for path in tgevp_outputs:
    print(Path(path).name)
